# Step 02. A* 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 01 Dijkstra는 **방사형 전방위 탐색**이라 목표와 무관한 노드도 많이 봤습니다.  
A\*는 **Heuristic** `h(n)`을 더해 목표 방향으로 집중 탐색합니다.

> Step 01처럼 작은 조각으로 나눠 이해하고, 마지막에 **Dijkstra와 나란히 비교**합니다.

## 학습 목표

1. Dijkstra와 A\* 간의 관계 이해: 우선순위 `g` -> `f = g + h` 한 줄 차이 (섹션 5, TODO 3)
2. heuristic을 **함수로 분리**해 여러 방식(Euclidean/Manhattan) 구현 (섹션 1, TODO 1·2)
3. heuristic **방식에 따라** 탐색 노드 수·경로 길이가 어떻게 달라지는지 비교 (섹션 6)
4. 이동 비용(대각선)이 '무엇이 최단인가'를 정의함을 확인 (섹션 7, TODO 4)

In [ ]:
%matplotlib inline
import sys, os, math, heapq, time
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../utils/01_global_planning'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from grid_map import MakeSimpleMap, BuildObstacleMap
from visualization import PlotGridMap, PlotSearchResult
from grid_search import (CalcGridIndex, CalcGridPosition, NodeKey,
                               GetMotionModel, IsValid, ReconstructPath)
from search_viz import AnimateSearch, AnimateCompare
from heuristic_viz import PlotHeuristicField

ox, oy, sx, sy, gx, gy = MakeSimpleMap()   # o*: 장애물(obstacle) 좌표, s*: 출발점(start), g*: 목표점(goal)
RESOLUTION, ROBOT_RADIUS = 2.0, 1.0
min_x, max_x = round(min(ox)), round(max(ox))
min_y, max_y = round(min(oy)), round(max(oy))
x_width = int((max_x-min_x)/RESOLUTION) + 1; y_width = int((max_y-min_y)/RESOLUTION) + 1   # BuildObstacleMap 과 동일

print('환경 로드 완료 (Step 01와 동일한 맵 -> 탐색 효율 직접 비교)')
fig, ax = plt.subplots(figsize=(6, 6))
PlotGridMap(ox, oy, sx, sy, gx, gy, ax=ax); ax.set_title('Search Environment')
plt.tight_layout(); plt.show()


## 1. Heuristic 함수

`h(n)` = 현재 → 목표 **예상** 비용. 아래 두 조건을 만족하면 **최적 경로**를 찾습니다.
- **admissible**: 과대추정 안 함 (남은 비용을 실제보다 크게 추정하지 않음)
- **consistent**: 한 칸 이동 시 h 감소량 ≤ 이동 비용 (확정한 칸(Closed)을 다시 보지 않는 이 코드에 필요)
- Euclidean(직선거리): 둘 다 만족 → 최적 / Manhattan(가로+세로 거리): 대각선에서 과대추정(대각선 1칸 실제 √2, Manhattan 2) → 최적 보장 X

> 참고 (심화, 건너뛰어도 됨)
> - 8방향 격자에서 실제 이동 비용과 딱 맞는 heuristic 은 octile 거리 `max(|dx|,|dy|) + (√2-1)·min(|dx|,|dy|)` (Euclidean 보다 실제 비용에 가까워 탐색 더 적음)

### 실습 (TODO 1·2)

여러 heuristic을 **함수로 분리**해 두고, 뒤에서 바꿔 끼우며 비교합니다.
가장 단순한 `HeuristicZero` 가 예시로 완성돼 있으니(이때 A\*는 Dijkstra와 동일), 나머지 `HeuristicEuclidean` · `HeuristicManhattan` 의 **TODO 1·2** 를 채워보세요. (비교 실습은 섹션 6)

In [ ]:
# === Heuristic 함수 모음 ===
# h(n): 현재 노드 -> 목표까지의 "예상" 비용.
#  - 최적 조건: admissible + consistent (Euclidean 은 둘 다 만족)
#  - 아래 함수들을 바꿔 끼우며(섹션 6 실습) 탐색이 어떻게 달라지는지 관찰한다.

def HeuristicZero(node, goal):
    """h(n)=0 -> A*가 Dijkstra와 동일해진다 (가장 단순한 베이스 예시)."""
    return 0.0

def HeuristicEuclidean(node, goal):
    """유클리드 거리 sqrt(dx^2 + dy^2)."""
    ##### TODO 1 #####
    # 유클리드 거리 계산하기: sqrt(dx^2 + dy^2)
    #   - dx = goal.x - node.x, dy = goal.y - node.y   (아직 없는 변수 -> 빈칸 위에 직접 만든 뒤 사용)
    # 사용함수: math.hypot(dx, dy) / 기능: sqrt(dx*dx + dy*dy)
    #################

    distance = ...

    #################
    return distance

def HeuristicManhattan(node, goal):
    """맨해튼 거리 |dx| + |dy|."""
    ##### TODO 2 #####
    # 맨해튼 거리 계산하기: |dx| + |dy|
    #   - dx, dy 는 위 HeuristicEuclidean 과 같은 방식으로 직접 계산
    # 사용함수: abs(x) / 기능: 절댓값 (파이썬 내장)
    # 주의: abs(dx) + abs(dy) 처럼 각각 절댓값 (abs(dx + dy) 는 부호가 다를 때 오답)
    #################

    distance = ...

    #################
    return distance

# --- 세 점 예시로 heuristic 값 비교 (실제 거리 5.0 과 비교) ---
class _P:
    def __init__(s, x, y): s.x, s.y = x, y
a, b = _P(0, 0), _P(3, 4)
print('Heuristic 비교 ((0,0) -> (3,4), 실제거리 5.0)')
print(f'  Zero      : {HeuristicZero(a, b):.2f}  (= Dijkstra, 비효율)')
print(f'  Euclidean : {HeuristicEuclidean(a, b):.2f} ')
print(f'  Manhattan : {HeuristicManhattan(a, b):.2f} ')

# --- heuristic 확인: 부호가 다른 방향까지 검사 (기대값 Euclidean 5.0 / Manhattan 7.0) ---
for p in (_P(3, 4), _P(-3, 4), _P(3, -4)):
    assert abs(HeuristicEuclidean(a, p) - 5.0) < 1e-9, f'TODO 1 확인 - Euclidean 오답: (0,0)->({p.x},{p.y}) 기대값 5.0'
    assert abs(HeuristicManhattan(a, p) - 7.0) < 1e-9, \
        f'TODO 2 확인 - Manhattan 오답: (0,0)->({p.x},{p.y}) 기대값 7.0 -> |dx| + |dy| 각각 절댓값 (abs(dx + dy) 아님)'
print('✅ heuristic 2개 통과')

## 2. Heuristic 필드

격자 전체의 `h(n)` 을 색으로 그리면 목표에 가까울수록 값이 작아지는(그림에서 **밝아지는**) **기울기**가 보입니다. (단위: 격자 칸)
A\*는 `g(n)`(실제 비용)에 이 `h(n)`을 더해, 목표 방향으로 탐색을 기울입니다.

**heuristic term이 없을 때(`HeuristicZero`)** 와 **있을 때(`HeuristicEuclidean`)** 를 나란히 비교합니다.
Zero는 기울기가 없어(균일) Dijkstra와 동일하고, Euclidean은 목표 방향으로 부드러운 기울기를 만듭니다.

In [ ]:
PlotHeuristicField(HeuristicZero, HeuristicEuclidean,
                   ox, oy, sx, sy, gx, gy,
                   min_x, max_x, min_y, max_y, x_width, y_width, RESOLUTION)

print('heuristic이 없으면 기울기가 없어 전방위 탐색(Dijkstra), 있으면 목표 방향으로 탐색이 기운다')

## 3. Node 클래스

Step 01와 동일한 Node (격자 인덱스 + 누적 비용 g + 부모 키).  
좌표변환·IsValid·역추적은 `grid_search`에서 그대로 재사용합니다.

In [ ]:
class Node:
    def __init__(self, x, y, cost, parent_index):
        self.x = x; self.y = y
        self.cost = cost                  # g(n): 출발 -> 현재 실제 비용
        self.parent_index = parent_index
    def __lt__(self, other):
        return self.cost < other.cost

print('Node 정의 완료 (좌표변환/IsValid/역추적은 grid_search 재사용)')


## 4. Dijkstra와의 단 한 줄 차이

알고리즘으로서의 차이는 우선순위 큐에 넣는 값 하나: `g` -> `g + h(n)`.
(구현은 한 함수로 합치면서 자료구조가 Step 01과 조금 다름 -> 섹션 5 표)

In [ ]:
# Dijkstra와 A*의 유일한 차이: 우선순위 큐에 넣는 값
print('Dijkstra:  heapq.heappush(pq, (g,        node))   # g(n)')
print('A*      :  heapq.heappush(pq, (g + h(n), node))   # f = g + h')
print()
print('알고리즘 차이는 이 한 줄 (구현 구조 차이는 섹션 5 표 참고)')


## 5. Astar() 조립

Step 01의 흐름(pop -> Closed 확정 -> 이웃 push)을 한 함수로 합치고, push 우선순위만 `f = g + h` 로 바꿉니다.
구현에서 Step 01과 달라진 점:

| | Step 01 | Step 02 (여기) |
|---|---|---|
| 노드 키 | `(ix, iy)` 튜플 | `NodeKey(node, ...)` -> `(ix, iy)` 를 하나의 정수로 변환 (`grid_search`) |
| Open Set | heap 만 (중복 push, pop 때 Closed 면 버림) | heap + `open_set` dict (더 싼 경로를 찾았을 때만 push) |
| 우선순위 | `g` | `f = g + h` |

`explore_order`도 반환해 동일하게 애니메이션에 사용합니다.

### 실습 (TODO 3)

A\*의 핵심 한 줄: 이웃을 우선순위 `f = g + h` 로 Open Set(`pq`)에 push 합니다.

In [ ]:
def Astar(sx, sy, gx, gy, ox, oy, resolution, robot_radius, heuristic_func=HeuristicZero, motion_model=None):
    if motion_model is None: motion_model = GetMotionModel()   # 기본: 직선 1.0 / 대각선 sqrt(2)

    obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = \
        BuildObstacleMap(ox, oy, resolution, robot_radius)
    start = Node(CalcGridIndex(sx, min_x, resolution), CalcGridIndex(sy, min_y, resolution), 0.0, -1)
    goal  = Node(CalcGridIndex(gx, min_x, resolution), CalcGridIndex(gy, min_y, resolution), 0.0, -1)

    open_set = {NodeKey(start, x_width, min_x, min_y): start}
    closed_set = {}
    pq = [(heuristic_func(start, goal), start)]      # f = g(0) + h
    explore_order = []

    while pq:
        _, current = heapq.heappop(pq)
        cid = NodeKey(current, x_width, min_x, min_y)   # cid: 현재 노드의 격자 키 (current id)
        if cid not in open_set: continue
        open_set.pop(cid); closed_set[cid] = current
        explore_order.append((CalcGridPosition(current.x, min_x, resolution),
                              CalcGridPosition(current.y, min_y, resolution), current.cost))
        if current.x == goal.x and current.y == goal.y:
            goal.parent_index = current.parent_index; goal.cost = current.cost; break
        for dx, dy, cost in motion_model:
            nb = Node(current.x+dx, current.y+dy, current.cost+cost, cid)   # nb: 이웃(neighbor) 노드
            nk = NodeKey(nb, x_width, min_x, min_y)   # nk: 이웃 노드의 격자 키 (neighbor key)
            if not IsValid(nb, obstacle_map, min_x, min_y, max_x, max_y, resolution): continue
            if nk in closed_set: continue
            if nk not in open_set or open_set[nk].cost > nb.cost:
                open_set[nk] = nb

                ##### TODO 3 #####
                # 이웃을 Open Set(pq) 에 push 하기: 우선순위 f = g + h  (A* 의 핵심, Dijkstra 는 f = g)
                #   - g : nb.cost (출발부터 누적 비용)
                #   - h : heuristic_func(nb, goal) (목표까지 추정 비용, 섹션 1)
                # 사용함수: heapq.heappush(pq, (f, nb)) / 기능: f 가 작은 노드부터 pop 되도록 push
                # 주의: f 자리에 g(nb.cost) 만 넣으면 Dijkstra 와 같아짐 -> 아래 확인에서 1,071개
                #################

                heapq.heappush(...)

                #################
    path_x, path_y = ReconstructPath(goal, closed_set, min_x, min_y, resolution)
    closed_x = [CalcGridPosition(n.x, min_x, resolution) for n in closed_set.values()]
    closed_y = [CalcGridPosition(n.y, min_y, resolution) for n in closed_set.values()]
    return path_x, path_y, closed_x, closed_y, explore_order

# 확인: Euclidean heuristic -> 탐색 노드 735개 / 경로 127.2 m (f 대신 g 만 넣으면 1,071개 = Dijkstra)
if RESOLUTION == 2.0 and ROBOT_RADIUS == 1.0:
    _px, _py, _cx, _cy, _ = Astar(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS, heuristic_func=HeuristicEuclidean)
    _len = sum(math.hypot(_px[i] - _px[i-1], _py[i] - _py[i-1]) for i in range(1, len(_px)))
    print('✅ Astar() 확인 통과: 735개 / 127.2 m' if (len(_cx) == 735 and abs(_len - 127.2) < 0.1) else
          f'❌ 탐색 {len(_cx):,}개 / {_len:.1f} m -> 기대값 735개 / 127.2 m, TODO 3 의 f = g + h 확인')


## 6. 실행 및 Heuristic 비교

섹션 1에서 구현한 `Heuristic*` 함수를 `Astar(..., heuristic_func=...)` 로 바꿔 끼우며,
**Zero(=Dijkstra) · Euclidean · Manhattan** 의 탐색 노드 수와 경로 길이를 한 번에 비교합니다.
- Zero: A\*가 Dijkstra와 완전히 같음 (Step 01과 같은 1,071개)
- Euclidean(admissible·consistent): 경로는 최적 유지, 탐색만 감소
- Manhattan(과대추정): 탐색은 더 줄지만 최적 **보장 X** -> 이 맵에서는 우연히 같은 길이, 과대추정을 크게 키우면(h × 10) 경로가 실제로 길어짐

In [ ]:
# 섹션 1의 heuristic 함수를 바꿔 끼우며 A* 실행 -> 탐색 노드 수 / 경로 길이 비교
def PathLength(px, py):
    return sum(math.hypot(px[i]-px[i-1], py[i]-py[i-1]) for i in range(1, len(px)))

runs = {
    'Zero (=Dijkstra)': HeuristicZero,
    'Euclidean'       : HeuristicEuclidean,
    'Manhattan'       : HeuristicManhattan,
}
res = {}
print(f"{'heuristic':<18}{'nodes':>9}{'path len':>11}")
print('-' * 38)
for name, hfunc in runs.items():
    px, py, cx, cy, order = Astar(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS, heuristic_func=hfunc)
    res[name] = (px, py, cx, cy, order)
    print(f"{name:<18}{len(cx):>9,}{PathLength(px, py):>11.2f}")

# 과대추정을 크게 키우면? h x 10 -> 탐색은 더 줄지만 경로가 최적(Zero 와 같은 길이)보다 길어짐
px10, py10, cx10, _, _ = Astar(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS,
                               heuristic_func=lambda node, goal: 10 * HeuristicEuclidean(node, goal))
print(f"{'10 x Euclidean':<18}{len(cx10):>9,}{PathLength(px10, py10):>11.2f}   <- 과대추정: 최적보다 김")

# 이후 섹션 8 애니메이션 / 섹션 9 그림에서 쓸 대표 결과: Dijkstra(zero) + A*(Euclidean)
d_px, d_py, d_cx, d_cy, d_order = res['Zero (=Dijkstra)']   # d_*: Dijkstra 결과 (path_x, path_y, closed_x, closed_y, explore_order)
path_x, path_y, closed_x, closed_y, explore_order = res['Euclidean']

names = list(runs.keys())
nodes = [len(res[n][2]) for n in names]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(names, nodes, color=['#4C72B0', '#55A868', '#C44E52'], alpha=0.9)
for i, v in enumerate(nodes): ax.text(i, v, f'{v:,}', ha='center', va='bottom')
ax.set_ylabel('explored nodes'); ax.set_title('Explored nodes by heuristic')
plt.tight_layout(); plt.show()

print()
print('관찰 포인트')
print(' - Zero      : h=0 -> Dijkstra와 동일 (탐색 노드 가장 많음, 경로는 최적)')
print(' - Euclidean : admissible·consistent -> 경로 길이 유지하며 탐색만 감소')
print(' - Manhattan : 대각선 과대추정 -> 탐색은 더 줄지만 최적 보장 X (이 맵에서는 같은 길이, h x 10 은 실제로 길어짐)')

## 7. 이동 비용(대각선) 실험

heuristic이 *탐색 효율*의 손잡이라면, **이동 비용(motion cost)** 은 *경로 모양*의 손잡이입니다.
비용 모델(`GetMotionModel`)이 곧 "무엇이 최단인가"의 정의라서, 대각선 비용을 어떻게 주느냐에 따라 최단 경로의 모양이 바뀝니다.

- **대각선 = √2** 일 때만 격자 비용이 실제 유클리드 거리와 일치 → 진짜 최단
- **대각선 < √2** (예: 1.0, Chebyshev): 대각선 과소평가 → 대각선 남용 → 기하학적으로 더 긴 경로
- **대각선 > 2×직선** (예: 3.0): 대각선이 직선 2번보다 비싸 → 대각선 회피 → 계단형(Manhattan)
  (정확히 2.0 이면 대각선 1번 = 직선 2번으로 비용이 같아, 어느 쪽을 골라도 최단이라 계단형이 보장되지 않습니다)

여기서는 비용 효과만 깨끗이 보려고 **heuristic을 Zero(=Dijkstra)로 고정** 합니다 (어떤 비용이든 항상 최적 경로).

### 실습 (TODO 4)

`MakeMotionModel` 의 **대각선 비용**을 채우고, 대각선 비용을 바꿔가며 경로 모양이 어떻게 달라지는지 관찰하세요.

In [ ]:
# 실습: 이동 비용을 바꿔 끼우며 A* 실행 -> 경로 "모양"이 어떻게 달라지는지 비교
# heuristic 은 Zero(=Dijkstra)로 고정 -> 어떤 비용이든 항상 최적 경로 -> 비용 효과만 분리해서 관찰

def MakeMotionModel(straight_cost=1.0, diagonal_cost=math.sqrt(2)):
    """8방향 이동 모델 [dx, dy, cost]. 직선=straight_cost, 대각선=diagonal_cost."""
    motion = [
        [1, 0, straight_cost], [-1, 0, straight_cost], [0, 1, straight_cost], [0, -1, straight_cost],   # 직선 4방향
    ]
    ##### TODO 4 #####
    # 대각선 4방향 [dx, dy, cost] 를 motion 에 추가하기 (대각선은 (dx, dy) 가 (±1, ±1), 비용은 diagonal_cost)
    # 사용함수: list.extend(리스트) / 기능: 리스트 뒤에 여러 원소를 한 번에 추가
    # 주의: 인자는 리스트 1개 -> motion.extend([[...], [...], [...], [...]]) (대괄호 두 겹, 4개를 따로 넣으면 TypeError)
    #################

    motion.extend(...)

    #################
    return motion

def PathLength(px, py):
    return sum(math.hypot(px[i]-px[i-1], py[i]-py[i-1]) for i in range(1, len(px)))

# 직선 비용은 1.0 고정, 대각선 비용만 바꿔가며 비교
diagonal_costs = [1.0, math.sqrt(2), 3.0]
labels = ['1.0 (Chebyshev)', 'sqrt2 (Euclidean)', '3.0 (-> Manhattan)']

fig, axes = plt.subplots(1, len(diagonal_costs), figsize=(4.2*len(diagonal_costs), 4.8))
print(f"{'diagonal cost':<20}{'path len':>10}{'#diagonals':>12}")
print('-' * 42)
for ax, dc, lab in zip(axes, diagonal_costs, labels):
    px, py, cx, cy, order = Astar(sx, sy, gx, gy, ox, oy, RESOLUTION, ROBOT_RADIUS,
                                  heuristic_func=HeuristicZero,
                                  motion_model=MakeMotionModel(1.0, dc))
    n_diag = sum(1 for i in range(1, len(px))
                 if abs(px[i]-px[i-1]) > 1e-6 and abs(py[i]-py[i-1]) > 1e-6)
    PlotGridMap(ox, oy, sx, sy, gx, gy, ax=ax)
    ax.plot(px, py, '-r', lw=2, label='Path')
    ax.set_title(f'diag = {lab}\nlen={PathLength(px, py):.1f}, diagonals={n_diag}', fontsize=9)
    print(f"{lab:<20}{PathLength(px, py):>10.1f}{n_diag:>12}")
plt.suptitle('heuristic=Zero: diagonal cost reshapes the optimal path (same map)', fontsize=13)
plt.tight_layout(); plt.show()

print()
print('sqrt2  : 격자 비용 = 실제 거리 -> 진짜 최단 (len 최소)')
print('1.0    : 대각선 과소평가 -> 대각선 남용 -> 기하학적으로 더 긴 경로')
print('3.0    : 대각선이 직선 2번보다 비싸 -> 대각선 회피 -> 계단형(Manhattan)')

## 8. 탐색 과정 동시 재생

왼쪽 Dijkstra(방사형) vs 오른쪽 A\*(목표 방향 집중)를 **같은 프레임에 나란히** 재생합니다.  
A\*의 탐색 영역이 목표 쪽으로 좁게 뻗는 것을 직접 비교하세요.

In [ ]:
# 두 알고리즘의 탐색 과정을 좌우로 동시 재생 (search_viz.AnimateCompare)
HTML(AnimateCompare(d_order, d_px, d_py, explore_order, path_x, path_y,
                     ox, oy, sx, sy, gx, gy, label_a='Dijkstra', label_b='A*',
                    n_batches=25, dpi=50).to_jshtml())   # dpi·프레임 수 축소 -> 노트북 용량 절감


## 9. 결과 요약

### A\* 특징

| 항목 | Dijkstra | A\* |
|------|----------|----|
| 우선순위 | g | g + h |
| 탐색 | 방사형 | 목표 집중 |
| 최적성 | 보장 | admissible·consistent h면 보장 |

→ Step 03 Hybrid A\*: 연속 상태 (x,y,yaw) + 차량 기구학

In [ ]:
fig = PlotSearchResult(closed_x, closed_y, path_x, path_y, ox, oy, sx, sy, gx, gy,
                         title=f'A* ({len(closed_x):,} nodes)')
plt.tight_layout(); plt.show()
print('A*: admissible·consistent heuristic(예: Euclidean)이면 Dijkstra와 같은 최적 경로를 더 적은 탐색으로')
print('다음 Step 03 Hybrid A*: 격자 (ix,iy) -> 연속 (x,y,yaw), Bicycle Model로 차량 주행 가능 경로')
